In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.config import PROCESSED, SPLITS
from src.cv import load_fold_table
from src.data import feature_cols
from src.evaluate import cross_validate

dev = pd.read_csv(PROCESSED / "dev_allclass.csv")
table = load_fold_table("cv_folds_allclass.csv")
groups = pd.read_csv(SPLITS / "groups_all.csv").set_index("segment")["group"]
hold_ids = set(pd.read_csv(SPLITS / "holdout_segments_allclass.csv")["segment"])

assert set(dev["segment"]) == set(table["segment"])
assert set(dev["segment"]).isdisjoint(hold_ids)
assert set(groups[list(hold_ids)]).isdisjoint(set(groups[dev["segment"].unique()]))
print("splits are consistent:", dev["segment"].nunique(), "dev segments,", len(hold_ids), "hold-out segments")
print("seizure share of dev segments:", round((dev.groupby("segment")["y"].first() == 1).mean(), 2))

cols = feature_cols(dev)
X = dev[cols].to_numpy(dtype=float)
log_amplitude = np.log10(X.std(axis=1, keepdims=True))                           # one number per chunk
three = np.c_[log_amplitude[:, 0], stats.kurtosis(X, axis=1), np.abs(np.diff(X, axis=1)).mean(axis=1)]

splits are consistent: 400 dev segments, 100 hold-out segments
seizure share of dev segments: 0.2


In [2]:
logistic = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
runs = {
    "chance (predicts the class prior)": (X, lambda: DummyClassifier(strategy="prior")),
    "one number: log amplitude": (log_amplitude, logistic),
    "three numbers: amplitude, kurtosis, line length": (three, logistic),
}
results = {name: cross_validate(make_model, dev, F, table, positive=1) for name, (F, make_model) in runs.items()}
summary = pd.DataFrame({name: r.drop(columns="repeat").mean() for name, r in results.items()}).T
summary[["auc_fold", "pr_auc_fold", "sensitivity", "specificity", "precision", "f1", "balanced_accuracy", "accuracy"]].round(3)

,auc_fold,pr_auc_fold,sensitivity,specificity,precision,f1,balanced_accuracy,accuracy
chance (predicts the class prior),0.500,0.200,0.000,1.000,0.000,0.000,0.500,0.800
one number: log amplitude,0.997,0.991,0.865,0.991,0.958,0.909,0.928,0.965
"three numbers: amplitude, kurtosis, line length",0.996,0.988,0.838,0.991,0.960,0.895,0.914,0.960


In [3]:
summary[["tp", "fn", "fp", "tn"]].round(1)

,tp,fn,fp,tn
chance (predicts the class prior),0.0,80.0,0.0,320.0
one number: log amplitude,69.2,10.8,3.0,317.0
"three numbers: amplitude, kurtosis, line length",67.0,13.0,2.8,317.2


# 09 Seizure baselines: takeaway

Seizure vs the rest, twin-grouped folds, 5 repeats, per-segment scores. 20% of segments are seizures.

| Model | AUC | PR-AUC | Sensitivity | Specificity | Accuracy |
|---|---|---|---|---|---|
| Chance | 0.500 | 0.200 | 0.000 | 1.000 | 0.800 |
| Log amplitude | 0.997 | 0.991 | 0.865 | 0.991 | 0.965 |
| Amplitude, kurtosis, line length | 0.996 | 0.988 | 0.838 | 0.991 | 0.960 |

- **It is easy.** One number, log amplitude, separates seizure from non-seizure almost perfectly (AUC 0.997).
- **More features add nothing.** Kurtosis and line length do not improve on amplitude.
- **Accuracy is misleading.** Chance scores 0.80 by finding no seizures.
- **The cut-off is the weak point.** At 0.5, about 11 of 80 seizure segments are missed, with only 3 false alarms. The ranking is good, but the cut-off is too strict.
- **AUC cannot tell models apart now,** because it is already near 1.

Caveats: chunk features and logistic regression only, dev data only, hold-out untouched.